In [1]:
import pandas as pd

In [3]:
df = pd.read_csv("files/atp_matches_2022.csv")
df

,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,match_num,winner_id,winner_seed,winner_entry,...,l_1stIn,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced,winner_rank,winner_rank_points,loser_rank,loser_rank_points
0,2022-8888,Atp Cup,Hard,16,A,20220103,300,200000,NaN,NaN,...,50.0,32.0,7.0,10.0,3.0,5.0,11.0,3308.0,19.0,2260.0
1,2022-8888,Atp Cup,Hard,16,A,20220103,299,133430,NaN,NaN,...,33.0,21.0,8.0,9.0,3.0,6.0,14.0,2475.0,20.0,2230.0
2,2022-8888,Atp Cup,Hard,16,A,20220103,298,105138,NaN,NaN,...,80.0,62.0,20.0,16.0,6.0,7.0,19.0,2260.0,9.0,3706.0
3,2022-8888,Atp Cup,Hard,16,A,20220103,297,105807,NaN,NaN,...,27.0,17.0,1.0,7.0,4.0,8.0,20.0,2230.0,860.0,18.0
4,2022-8888,Atp Cup,Hard,16,A,20220103,296,106421,NaN,NaN,...,35.0,22.0,4.0,8.0,3.0,7.0,2.0,8640.0,11.0,3308.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2912,2022-M-DC-2022-WG2-PO-GRE-JAM-01,Davis Cup WG2 PO: GRE vs JAM,Clay,4,D,20220304,4,209362,NaN,NaN,...,68.0,42.0,12.0,10.0,11.0,13.0,1103.0,9.0,1130.0,8.0
2913,2022-M-DC-2022-WG2-PO-GRE-JAM-01,Davis Cup WG2 PO: GRE vs JAM,Clay,4,D,20220304,5,202065,NaN,NaN,...,56.0,40.0,20.0,15.0,4.0,8.0,808.0,23.0,1390.0,4.0
2914,2022-M-DC-2022-WG2-PO-HKG-BEN-01,Davis Cup WG2 PO: HKG vs BEN,Hard,4,D,20220304,1,138846,NaN,NaN,...,54.0,29.0,8.0,11.0,6.0,10.0,1059.0,10.0,1881.0,1.0
2915,2022-M-DC-2022-WG2-PO-HKG-BEN-01,Davis Cup WG2 PO: HKG vs BEN,Hard,4,D,20220304,2,209409,NaN,NaN,...,39.0,24.0,7.0,10.0,5.0,9.0,1050.0,10.0,NaN,NaN


In [4]:
# Step 1: Create player-level rows for both winner and loser
winner_df = df.rename(columns={
    'winner_name': 'player_name',
    'w_ace': 'ace',
    'w_df': 'df',
    'w_1stWon': 'first_serve_won',
    'w_2ndWon': 'second_serve_won',
    'w_bpSaved': 'bp_saved',
    'w_svpt': 'serve_pts',
    'surface': 'surface'
})
loser_df = df.rename(columns={
    'loser_name': 'player_name',
    'l_ace': 'ace',
    'l_df': 'df',
    'l_1stWon': 'first_serve_won',
    'l_2ndWon': 'second_serve_won',
    'l_bpSaved': 'bp_saved',
    'l_svpt': 'serve_pts',
    'surface': 'surface'
})

# Add a 'player_result' column if needed
winner_df['player_result'] = 1
loser_df['player_result'] = 0

# Combine both
player_df = pd.concat([winner_df[['player_name', 'surface', 'ace', 'df', 'first_serve_won',
                                  'second_serve_won', 'bp_saved', 'serve_pts']],
                       loser_df[['player_name', 'surface', 'ace', 'df', 'first_serve_won',
                                 'second_serve_won', 'bp_saved', 'serve_pts']]], ignore_index=True)


In [5]:
player_df

,player_name,surface,ace,df,first_serve_won,second_serve_won,bp_saved,serve_pts
0,Felix Auger Aliassime,Hard,15.0,6.0,38.0,14.0,10.0,78.0
1,Denis Shapovalov,Hard,7.0,2.0,34.0,16.0,8.0,78.0
2,Roberto Bautista Agut,Hard,1.0,2.0,50.0,20.0,1.0,96.0
3,Pablo Carreno Busta,Hard,6.0,0.0,25.0,8.0,0.0,45.0
4,Daniil Medvedev,Hard,6.0,4.0,22.0,10.0,0.0,41.0
...,...,...,...,...,...,...,...,...
5829,Blaise Bicknell,Clay,1.0,3.0,42.0,12.0,11.0,96.0
5830,Rowland Phillips,Clay,1.0,1.0,40.0,20.0,4.0,94.0
5831,Alexis Klegou,Hard,0.0,3.0,29.0,8.0,6.0,72.0
5832,Delmas Ntcha,Hard,1.0,5.0,24.0,7.0,5.0,58.0


In [6]:
# Avoid division by zero
player_df = player_df[player_df['serve_pts'] > 0]

# Compute rates
player_df['ace_rate'] = player_df['ace'] / player_df['serve_pts']
player_df['df_rate'] = player_df['df'] / player_df['serve_pts']
player_df['first_serve_win_pct'] = player_df['first_serve_won'] / player_df['serve_pts']
player_df['second_serve_win_pct'] = player_df['second_serve_won'] / player_df['serve_pts']
player_df['bp_saved_pct'] = player_df['bp_saved'] / player_df['serve_pts']

# Aggregate per player per surface
player_surface_stats = player_df.groupby(['player_name', 'surface']).agg({
    'ace_rate': 'mean',
    'df_rate': 'mean',
    'first_serve_win_pct': 'mean',
    'second_serve_win_pct': 'mean',
    'bp_saved_pct': 'mean'
}).reset_index()
player_surface_stats

/var/folders/3t/356b2m1s0713vmrmjlw72vfm0000gn/T/ipykernel_44580/4126321600.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  player_df['ace_rate'] = player_df['ace'] / player_df['serve_pts']
/var/folders/3t/356b2m1s0713vmrmjlw72vfm0000gn/T/ipykernel_44580/4126321600.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  player_df['df_rate'] = player_df['df'] / player_df['serve_pts']
/var/folders/3t/356b2m1s0713vmrmjlw72vfm0000gn/T/ipykernel_44580/4126321600.py:7: SettingWithCopyWarning: 
A value is trying t

,player_name,surface,ace_rate,df_rate,first_serve_win_pct,second_serve_win_pct,bp_saved_pct
0,Adrian Mannarino,Clay,0.034560,0.031411,0.361999,0.211839,0.056379
1,Adrian Mannarino,Grass,0.075787,0.034631,0.444298,0.200393,0.045524
2,Adrian Mannarino,Hard,0.077486,0.028034,0.437833,0.217900,0.038791
3,Aisam Ul Haq Qureshi,Grass,0.092308,0.076923,0.492308,0.261538,0.000000
4,Alastair Gray,Grass,0.052642,0.039821,0.473679,0.157974,0.066871
...,...,...,...,...,...,...,...
657,Zdenek Kolar,Grass,0.024096,0.048193,0.421687,0.132530,0.060241
658,Zhizhen Zhang,Hard,0.079644,0.014083,0.457135,0.201151,0.035154
659,Zizou Bergs,Grass,0.072165,0.020619,0.474227,0.175258,0.020619
660,Zizou Bergs,Hard,0.079273,0.012235,0.505809,0.157562,0.055646


In [7]:
from sklearn.preprocessing import StandardScaler

normalized_data = []
scalers = {}

for surface in player_surface_stats['surface'].unique():
    temp = player_surface_stats[player_surface_stats['surface'] == surface].copy()
    features = temp.drop(columns=['player_name', 'surface'])
    
    scaler = StandardScaler()
    temp_scaled = scaler.fit_transform(features)
    scalers[surface] = scaler  # Save if needed later

    temp[features.columns] = temp_scaled
    normalized_data.append(temp)

surface_normalized = pd.concat(normalized_data, ignore_index=True)


In [8]:
from sklearn.cluster import KMeans

clusters = []
kmeans_models = {}

for surface in surface_normalized['surface'].unique():
    temp = surface_normalized[surface_normalized['surface'] == surface].copy()
    features = temp.drop(columns=['player_name', 'surface'])

    kmeans = KMeans(n_clusters=3, random_state=42)
    temp['cluster'] = kmeans.fit_predict(features)

    kmeans_models[surface] = kmeans
    clusters.append(temp)

surface_clustered = pd.concat(clusters, ignore_index=True)
surface_clustered

,player_name,surface,ace_rate,df_rate,first_serve_win_pct,second_serve_win_pct,bp_saved_pct,cluster
0,Adrian Mannarino,Clay,-0.405161,-0.196572,-0.992938,0.764584,-0.014705,1
1,Albert Ramos,Clay,-0.334505,-0.646726,0.416784,0.030878,-0.313078,0
2,Alejandro Davidovich Fokina,Clay,-0.649360,0.109163,0.738024,-0.685419,-0.228988,2
3,Alejandro Tabilo,Clay,0.291700,0.007552,0.467792,-0.267613,0.009747,0
4,Alessandro Giannessi,Clay,-0.204366,0.277824,0.629633,-0.376827,2.037029,2
...,...,...,...,...,...,...,...,...
657,Yuta Shimizu,Hard,-1.244633,0.716200,-3.118961,0.735784,-0.859095,2
658,Zachary Svajda,Hard,-1.648325,-1.939107,0.093568,-0.796043,-0.912101,2
659,Zhizhen Zhang,Hard,0.216478,-1.216131,0.531217,0.564350,-0.828135,0
660,Zizou Bergs,Hard,0.207775,-1.310988,1.426215,-0.736167,0.116841,0


In [10]:
surface_clustered[['player_name', 'surface', 'cluster']].sort_values(by=['player_name', 'surface']).to_csv("surface_clusters_2022.csv")